Silver reproducibility notebook

- Configure `DATA_DOI`, `MODEL_DOI`, filenames, and switches in the next cell.
- Loads data from local `Data/` or downloads from Zenodo, verifies optional hashes, sets deterministic seeds, trains or loads a PyTorch model, captures environment.

In [1]:
# Configuration and reproducibility helpers
from __future__ import annotations
import os, sys, json, hashlib, random, pathlib
from typing import Optional
import numpy as np

# Bronze DOIs used by default
DATA_DOI = os.environ.get("DATA_DOI", "10.5281/zenodo.17298664")
MODEL_DOI = os.environ.get("MODEL_DOI", "10.5281/zenodo.17298751")
DATA_FILENAME = os.environ.get("DATA_FILENAME", "simple_dataset.csv")
MODEL_FILENAME = os.environ.get("MODEL_FILENAME", "linear_model.pt")
ARTIFACTS_DIR = os.environ.get("ARTIFACTS_DIR", "artifacts")
DATA_DIR = os.environ.get("DATA_DIR", "Data")
VERBOSE = True
USE_DOWNLOADED_MODEL = os.environ.get("USE_DOWNLOADED_MODEL", "0") == "1"

EXPECTED_DATA_SHA256 = os.environ.get("EXPECTED_DATA_SHA256", "")
EXPECTED_MODEL_SHA256 = os.environ.get("EXPECTED_MODEL_SHA256", "")

GLOBAL_SEED = int(os.environ.get("GLOBAL_SEED", "674"))
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)

# Utils
def ensure_dir(path: str) -> None:
    pathlib.Path(path).mkdir(parents=True, exist_ok=True)

def sha256_of_file(path: str) -> str:
    sha = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            sha.update(chunk)
    return sha.hexdigest()

def verify_hash(path: str, expected_sha256: str) -> bool:
    if not expected_sha256:
        return True
    actual = sha256_of_file(path)
    if VERBOSE:
        print(f"SHA256 for {path}: {actual}")
    return actual.lower() == expected_sha256.lower()

def zenodo_download(doi: str, filename: str, dest_dir: str) -> str:
    import urllib.request
    ensure_dir(dest_dir)
    record_id = doi.split(".")[-1].replace("zenodo/", "").replace("zenodo-", "").replace("zenodo", "").replace("/", "")
    url = f"https://zenodo.org/records/{record_id}/files/{filename}?download=1"
    dest_path = os.path.join(dest_dir, filename)
    if VERBOSE:
        print(f"Downloading {url} -> {dest_path}")
    urllib.request.urlretrieve(url, dest_path)
    return dest_path

ensure_dir(ARTIFACTS_DIR)
ensure_dir(DATA_DIR)
print("Configuration loaded. Seeds set.")

Configuration loaded. Seeds set.


In [2]:
import os, pandas as pd
# Load dataset from local Data/ or download from Zenodo
local_path = os.path.join(DATA_DIR, DATA_FILENAME)
if not os.path.exists(local_path):
    if not DATA_DOI:
        raise RuntimeError("DATA_DOI is empty and local data not found.")
    local_path = zenodo_download(DATA_DOI, DATA_FILENAME, DATA_DIR)
    if EXPECTED_DATA_SHA256 and not verify_hash(local_path, EXPECTED_DATA_SHA256):
        raise RuntimeError("Downloaded data hash mismatch; refusing to proceed.")
print(f"Using dataset at: {local_path}")
df = pd.read_csv(local_path)
print(df.head())


Using dataset at: Data/simple_dataset.csv
   UserID  Age      City  Salary  Score
0       1   56   Chicago   91717  55.16
1       2   46   Chicago   95859  59.17
2       3   32  New York   71309  51.28
3       4   60   Chicago  108734  71.19
4       5   25   Chicago  115467  54.51


In [3]:
# Preview loaded data
print(df.head())

   UserID  Age      City  Salary  Score
0       1   56   Chicago   91717  55.16
1       2   46   Chicago   95859  59.17
2       3   32  New York   71309  51.28
3       4   60   Chicago  108734  71.19
4       5   25   Chicago  115467  54.51


In [7]:
import os, pickle
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

# Preprocess
df_processed = pd.get_dummies(df, columns=['City'], drop_first=True)
X = df_processed.drop(['UserID', 'Score'], axis=1).values
y = df_processed['Score'].values.reshape(-1, 1)

# Deterministic split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=GLOBAL_SEED
)

# Torch tensors
X_train_t = torch.tensor(X_train.astype(np.float32))
y_train_t = torch.tensor(y_train.astype(np.float32))
X_test_t = torch.tensor(X_test.astype(np.float32))
y_test_t = torch.tensor(y_test.astype(np.float32))

# Simple linear model
class LinearRegressionModel(nn.Module):
    def __init__(self, input_size: int, output_size: int):
        super().__init__()
        self.linear = nn.Linear(input_size, output_size)
    def forward(self, x):
        return self.linear(x)

model = LinearRegressionModel(X_train.shape[1], 1)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=5e-4)

# Train
num_epochs = 10000
for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()
    preds = model(X_train_t)
    loss = criterion(preds, y_train_t)
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 20 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")

# Evaluate
model.eval()
with torch.no_grad():
    y_pred_t = model(X_test_t)

y_pred = y_pred_t.numpy()
y_true = y_test_t.numpy()

mse = mean_squared_error(y_true, y_pred)
r2 = r2_score(y_true, y_pred)
print("--- Model Performance (PyTorch) ---")
print(f"MSE: {mse:.2f}")
print(f"R2: {r2:.2f}")

# Save model artifact
ensure_dir(ARTIFACTS_DIR)
model_path = os.path.join(ARTIFACTS_DIR, MODEL_FILENAME)
torch.save(model.state_dict(), model_path)
print(f"Saved torch state_dict to {model_path}")
print(f"Model SHA256: {sha256_of_file(model_path)}")


Epoch [20/10000], Loss: 324543264.0000
Epoch [40/10000], Loss: 295457856.0000
Epoch [60/10000], Loss: 268305968.0000
Epoch [80/10000], Loss: 243072032.0000
Epoch [100/10000], Loss: 219684736.0000
Epoch [120/10000], Loss: 198060640.0000
Epoch [140/10000], Loss: 178114640.0000
Epoch [160/10000], Loss: 159761872.0000
Epoch [180/10000], Loss: 142918304.0000
Epoch [200/10000], Loss: 127500560.0000
Epoch [220/10000], Loss: 113426456.0000
Epoch [240/10000], Loss: 100615232.0000
Epoch [260/10000], Loss: 88987496.0000
Epoch [280/10000], Loss: 78465600.0000
Epoch [300/10000], Loss: 68973840.0000
Epoch [320/10000], Loss: 60438552.0000
Epoch [340/10000], Loss: 52788452.0000
Epoch [360/10000], Loss: 45954740.0000
Epoch [380/10000], Loss: 39871244.0000
Epoch [400/10000], Loss: 34474688.0000
Epoch [420/10000], Loss: 29704694.0000
Epoch [440/10000], Loss: 25504026.0000
Epoch [460/10000], Loss: 21818572.0000
Epoch [480/10000], Loss: 18597502.0000
Epoch [500/10000], Loss: 15793218.0000
Epoch [520/10000]

In [ ]:
# Capture environment for Silver
import subprocess
ensure_dir(ARTIFACTS_DIR)
req_path = os.path.join(ARTIFACTS_DIR, "requirements-silver.txt")
freeze = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
with open(req_path, "w") as f:
    f.write(freeze)
print(f"Saved environment to {req_path}")
